# [실습 5-1] 나이브 베이즈 스팸 분류 — 손계산 재현에서 scikit-learn까지

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 30~40분 (CPU 충분) |
| 본문 연계 | 5.3 나이브 베이즈 |
| 선수 실습 | 없음 (실습 4-1 권장) |
| 준비 | 부록 B.1·B.3 참고 |

본문 5.3.1의 **손계산을 코드로 재현**하고, 같은 원리를
코퍼스 전체로 일반화한 뒤, scikit-learn과 결과를 대조한다 —
"라이브러리는 마법이 아니라 같은 계산"임을 확인하는 3단 상승 구조다.

### [준비] 환경 설정 (저장소 전용)

In [1]:
# 저장소 루트를 임포트 경로에 추가
# (Colab에서는 아래 두 줄의 주석을 해제하고 실행)
# !git clone https://github.com/tbgoodlife/ai-labs.git
# %cd ai-labs
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent          # ch05/에서 연 경우
sys.path.insert(0, str(ROOT))

import platform
import pandas as pd
import sklearn
from utils import plot_style

plot_style.apply()              # 도해 스타일 킷 적용
print("Python", platform.python_version())
print("pandas", pd.__version__,
      "/ scikit-learn", sklearn.__version__)

Python 3.12.6
pandas 3.0.3 / scikit-learn 1.9.0


### [셀 1] 1단 · 손계산 재현 — 미니 데이터와 우도표 📖

In [2]:
# 본문 5.3.1 미니 데이터 — 스팸 4통, 정상 6통
SPAM = ["무료 쿠폰 당첨 안내", "무료 이벤트 당첨 확인",
        "무료 상품권 지급", "경품 당첨 축하 회의 초대"]
HAM = ["내일 회의 자료 공유", "회의 일정 변경 안내",
       "점심 회의 예약", "프로젝트 회의 기록 전달",
       "무료 주차 등록 안내", "동호회 경품 당첨 결과"]
VOCAB = ["무료", "당첨", "회의"]

def likelihood(word, msgs):
    """단어가 등장한 메일의 비율 = P(단어|분류)."""
    hit = sum(word in m.split() for m in msgs)
    return hit / len(msgs)

print(f"{'단어':4}{'P(단어|스팸)':>12}{'P(단어|정상)':>12}")
for w in VOCAB:
    print(f"{w:<6}{likelihood(w, SPAM):>10.3f}"
          f"{likelihood(w, HAM):>12.3f}")

단어      P(단어|스팸)    P(단어|정상)
무료         0.750       0.167
당첨         0.750       0.167
회의         0.250       0.667


**핵심 포인트**
- 우도표가 곧 나이브 베이즈의 "학습 결과"다 — 분류별로 단어 비율을 세는 것이 전부다.
- 본문 5.3.1의 우도표와 같은 값이 나온다: 무료는 스팸의 단서(0.75 vs 0.167), 회의는 정상의 단서(0.25 vs 0.667).

기대 출력: 무료 0.750/0.167, 당첨 0.750/0.167, 회의 0.250/0.667

### [셀 2] 판정 함수 — 사전 확률 × 우도 곱 📖

In [3]:
def spam_score(message, spam_msgs, ham_msgs):
    """P(스팸|메시지)를 베이즈 정리로 계산한다."""
    words = [w for w in message.split() if w in VOCAB]
    n = len(spam_msgs) + len(ham_msgs)
    s = len(spam_msgs) / n          # 사전 확률 P(스팸)
    h = len(ham_msgs) / n           # 사전 확률 P(정상)
    for w in words:                 # 단어마다 우도를 곱함
        s *= likelihood(w, spam_msgs)
        h *= likelihood(w, ham_msgs)
    return s / (s + h)              # 정규화 → 사후 확률

msg = "무료 당첨 안내"
p = spam_score(msg, SPAM, HAM)
print(f"'{msg}' → 스팸일 확률 {p:.1%}")

'무료 당첨 안내' → 스팸일 확률 93.1%


**핵심 포인트**
- 계산 순서가 본문 5.3.1 진행표와 같다: 사전 확률에서 출발해 단어 우도를 하나씩 곱한다(단어 독립 가정 = "나이브").
- 우도표에 없는 단어("안내")는 건너뛴다.
- 마지막 나눗셈이 두 점수를 **확률**로 바꾼다 — 도입 질문의 "확률로 말하는 필터".

기대 출력: `'무료 당첨 안내' → 스팸일 확률 93.1%` — 본문 손계산과 동일 수치

실패 시 대처: 값이 다르면 [셀 1]의 메시지 목록이 수정되지 않았는지 확인한다.

### [셀 3] 2단 · 일반화 — 코퍼스 전체 학습(라플라스 평활) 📖

In [4]:
df = pd.read_csv(ROOT / "data" / "spam_mini.csv")
print("코퍼스:", df["label"].value_counts().to_dict())

def train_nb(texts, labels, alpha=1):
    """단어 빈도 기반 나이브 베이즈 학습.
    alpha: 라플라스 평활 가산값(기본 1)."""
    vocab = sorted({w for t in texts for w in t.split()})
    classes = sorted(set(labels))
    counts = {c: dict.fromkeys(vocab, 0) for c in classes}
    totals = dict.fromkeys(classes, 0)
    priors = {}
    for c in classes:
        docs = [t for t, l in zip(texts, labels)
                if l == c]
        priors[c] = len(docs) / len(texts)
        for t in docs:
            for w in t.split():
                counts[c][w] += 1
                totals[c] += 1
    V = len(vocab)
    like = {c: {w: (counts[c][w] + alpha)
                   / (totals[c] + alpha * V)
                for w in vocab} for c in classes}
    return priors, like

priors, like = train_nb(df["text"], df["label"])
print(f"어휘 {len(like['spam'])}개 학습 완료, "
      f"사전 확률 {priors}")

코퍼스: {'spam': 25, 'ham': 25}
어휘 114개 학습 완료, 사전 확률 {'ham': 0.5, 'spam': 0.5}


**핵심 포인트**
- [셀 1]의 "비율 세기"가 그대로 일반화됐다 — 등장 여부 대신 **빈도**를 세고, 분자·분모에 평활값을 더한다.
- **라플라스 평활**(`alpha=1`): 한 번도 안 나온 단어의 확률이 0이 되어 곱셈 전체를 무너뜨리는 것을 막는다(자세한 붕괴 실험은 [심화 1]).

### [셀 4] 3단 · scikit-learn과 대조 📖

In [5]:
from sklearn.feature_extraction.text import (
    CountVectorizer)
from sklearn.naive_bayes import MultinomialNB

def predict(text):
    """[셀 3] 학습 결과로 스팸 확률을 계산한다."""
    s = dict(priors)
    for w in text.split():
        if w in like["spam"]:       # 어휘 밖 단어는 무시
            for c in s:
                s[c] *= like[c][w]
    return s["spam"] / (s["spam"] + s["ham"])

vec = CountVectorizer(tokenizer=str.split,
                      token_pattern=None)
X = vec.fit_transform(df["text"])
clf = MultinomialNB(alpha=1).fit(X, df["label"])
spam_i = list(clf.classes_).index("spam")

TESTS = ["무료 쿠폰 당첨 안내 클릭",
         "내일 회의 자료 검토 부탁",
         "당첨 결과 회의 공지"]
for t in TESTS:
    lib = clf.predict_proba(vec.transform([t]))[0]
    print(f"'{t}'")
    print(f"  직접 구현 {predict(t):.1%} / "
          f"scikit-learn {lib[spam_i]:.1%}")

'무료 쿠폰 당첨 안내 클릭'
  직접 구현 99.8% / scikit-learn 99.8%
'내일 회의 자료 검토 부탁'
  직접 구현 0.1% / scikit-learn 0.1%
'당첨 결과 회의 공지'
  직접 구현 10.8% / scikit-learn 10.8%


**핵심 포인트**
- 직접 구현과 `MultinomialNB`의 확률이 **일치**한다 — 같은 평활(`alpha=1`), 같은 토큰화(공백 분리)이므로 같은 계산이다.
- 세 번째 문장(당첨+회의 혼재)은 두 단서가 줄다리기한다 — 확률이 극단에서 멀어진다.
- 정오 판정의 품질을 지표로 따지는 방법(정밀도·재현율)은 **6장**에서 다룬다.

실패 시 대처: 두 값이 다르면 `alpha`와 토큰화 기준이 양쪽에서 같은지 대조한다. `data/` 파일 없음 오류는 저장소 루트 기준 경로인지 확인한다.

### [보조 1] 코퍼스 훑어보기

In [6]:
for c in ["spam", "ham"]:
    words = pd.Series(
        [w for t, l in zip(df["text"], df["label"])
         if l == c for w in t.split()])
    top = words.value_counts().head(5)
    print(f"[{c}] 상위 단어:",
          ", ".join(f"{w}({n})" for w, n in top.items()))
print()
print(df.groupby("label").head(2).to_string(index=False))

[spam] 상위 단어: 무료(17), 당첨(11), 확인(6), 이벤트(5), 마감(5)
[ham] 상위 단어: 회의(8), 안내(6), 공지(6), 일정(5), 드림(5)

label              text
 spam 무료 쿠폰 당첨 안내 지금 확인
 spam   무료 이벤트 당첨 확인 요망
  ham    내일 회의 자료 공유 부탁
  ham    회의 일정 변경 안내 확인


### [보조 2] 언더플로 방지 — 로그 확률 버전

In [7]:
import math

def predict_log(text):
    """확률 곱셈 → 로그 덧셈: 단어가 수백 개여도 안전."""
    s = {c: math.log(priors[c]) for c in priors}
    for w in text.split():
        if w in like["spam"]:
            for c in s:
                s[c] += math.log(like[c][w])
    m = max(s.values())             # 오버플로 방지 트릭
    es = math.exp(s["spam"] - m)
    eh = math.exp(s["ham"] - m)
    return es / (es + eh)

t = TESTS[0]
print(f"곱셈 {predict(t):.4f} vs 로그 {predict_log(t):.4f}")

곱셈 0.9983 vs 로그 0.9983


실전 필터는 단어 수천 개의 확률을 곱한다 — 0.001을 천 번 곱하면 컴퓨터가 표현할 수 없이 작아진다(언더플로). 로그를 취해 **곱셈을 덧셈으로** 바꾸는 것이 표준 처방이며, scikit-learn 내부도 로그로 계산한다.

### [심화 1] 라플라스 평활 유무 비교 (연습문제 심화 직결)

In [8]:
# 평활 없이(alpha=0) 학습하면 무슨 일이 생길까?
_, like0 = train_nb(df["text"], df["label"], alpha=0)

def predict_with(like_t, text):
    s = dict(priors)
    for w in text.split():
        if w in like_t["spam"]:
            for c in s:
                s[c] *= like_t[c][w]
    total = s["spam"] + s["ham"]
    return s["spam"] / total if total else float("nan")

CASES = ["휴가 일정 무료 안내",   # '휴가'는 정상에만 등장
         "무료 쿠폰 회의 안내"]   # '쿠폰'은 스팸에만 등장
print(f"{'문장':<16} 평활O    평활X")
for t in CASES:
    a = predict_with(like, t)
    b = predict_with(like0, t)
    print(f"'{t}' {a:>7.1%} {b:>8.1%}")

문장               평활O    평활X
'휴가 일정 무료 안내'   29.0%     0.0%
'무료 쿠폰 회의 안내'   81.3%   100.0%


**의도된 붕괴다**: 평활이 없으면 한쪽 분류에서 한 번도 안 나온 단어 하나가 확률 전체를 0으로 만들어, 판정이 0% 또는 100%로 **극단화**된다. 다른 모든 단서를 단어 하나가 거부권으로 뒤집는 셈이다 — 연습문제 심화 문항에서 이 결과를 해석해 보자.

---
## 마무리

- 나이브 베이즈의 학습은 "분류별 단어 비율 세기", 판정은 "사전 확률 × 우도 곱" — 본문 손계산과 코드가 같은 계산이다.
- 라플라스 평활은 확률 0의 거부권을 막는 최소한의 안전장치다.
- 직접 구현 = scikit-learn 결과 일치 — 라이브러리를 **원리를 아는 채로** 쓸 수 있게 되었다.

**연습문제 연계**: [응용] 새 미니 데이터 나이브 베이즈 판정은 [셀 1]~[셀 2] 재실행으로 자가 채점, [심화] 평활 유무 실험 해석은 [심화 1]에서 수행한다.

**다음 실습**: [실습 6-1] 과적합 시각화 (`ch06/lab-06-01_overfitting-curves.ipynb`)